In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [2]:
load_dotenv()
lm = ChatOpenAI(model="openai/gpt-4o", base_url="https://openrouter.ai/api/v1", max_tokens=500,streaming=True)

In [3]:
ans = lm.invoke("what is India's national flower")
print(ans.content)

India's national flower is the lotus (Nelumbo nucifera). The lotus is a symbol of purity, beauty, wealth, fertility, and enlightenment in Indian culture and is respected in various religious and cultural traditions across the country.


STREAM RESPONSES


In [4]:
ans = lm.stream("What is Mango?")
# object
ans 
# realtime 
for chunk in ans:
    print(chunk," ")
# do not jump to new line
for chunk in ans:
    print(chunk.content,end=" ")


content='"M' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019ceadb-006e-7fd2-8028-5d22fda7c4bc' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]  
content='ango' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019ceadb-006e-7fd2-8028-5d22fda7c4bc' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]  
content='"' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019ceadb-006e-7fd2-8028-5d22fda7c4bc' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]  
content=' can' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019ceadb-006e-7fd2-8028-5d22fda7c4bc' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]  
content=' refer' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019ceadb-006e-7fd2-8028-5d22fda7c4bc' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]  
content=' to' additional_kwargs={} response_meta

In [4]:
prompts = [("system","You are a python developer"),
           ("user","how to define a function") ]
res = lm.invoke(prompts)
print(res.content)

In Python, you define a function using the `def` keyword followed by the function name and parentheses `()`. If the function requires parameters, you enclose them within the parentheses. The function body starts with a colon `:` and is indented. You can optionally include a `return` statement to return a value from the function. Here's a simple example:

```python
def my_function():
    print("Hello, World!")

my_function()  # This will call the function and print "Hello, World!"
```

Here's an example of a function with parameters:

```python
def add_numbers(a, b):
    return a + b

result = add_numbers(3, 4)
print(result)  # This will print 7
```

In this example:
- `add_numbers` is the function's name.
- `a` and `b` are parameters that you pass into the function.
- `return a + b` returns the sum of `a` and `b` to wherever the function was called.


PROMPT TEMPLATES


In [5]:
from langchain_core.prompts import ChatPromptTemplate

In [6]:
prompts = ChatPromptTemplate.from_messages(
          [{"role":"system","content":"translate given query into {language}"},
            {"role":"user","content":"{query}"}])
final = prompts.format_messages(language = "hindi", query = "I am a girl")
ans = lm.invoke(final)
ans.content

'मैं एक लड़की हूँ।'

In [7]:
final = prompts.format_messages(language = "hindi", query = "I am a girl")
ans = lm.invoke(final)
ans.content

'मैं एक लड़की हूँ।'

Runnable

In [8]:
final =lm.invoke(prompts.invoke({"language" : "hindi", "query" : "I am a girl"}))
final.content

'मैं एक लड़की हूँ।'

Chaining

In [9]:
chain = prompts | lm
res = chain.invoke({"language" : "hindi", "query" : "I am a girl"})
res.content

'मैं एक लड़की हूँ।'

Outputparser - pass string content



In [10]:
from langchain_core.output_parsers import StrOutputParser

out = StrOutputParser()
chain = prompts | lm | out
res = chain.invoke({"language" : "hindi", "query" : "I am a girl"})
res

'मैं एक लड़की हूँ।'

MEMORY MANAGEMENT

In [ ]:
history = []
while True:
    question = input("user:")
    history.append({"role":"user","content":question})
    res = lm.invoke(history)
    history.append({"role":"ai","content":res.content})
    if question.lower() in["bye","goodbye"]:
        print("Bye")
        break

STRUCTURED OUTPUT


In [11]:
from pydantic import BaseModel,Field
from typing import List

class Movies(BaseModel):
    name:str = Field(description="Name of Movie")
    year:int = Field(description="release year")
    movies: List[str] = Field(description="List of movies")

struct_lm = lm.with_structured_output(Movies)
res = struct_lm.invoke("give three bollywood movie")
res.model_dump()

C:\Users\Asus\AppData\Roaming\Python\Python313\site-packages\pydantic\main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [input_value=Movies(name='Dilwale Dulh...ali 2: The Conclusion']), input_type=Movies])
  return self.__pydantic_serializer__.to_python(


{'name': 'Dilwale Dulhania Le Jayenge',
 'year': 1995,
 'movies': ['Dilwale Dulhania Le Jayenge',
  'Kabhi Khushi Kabhie Gham',
  'Baahubali 2: The Conclusion']}